In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler , MinMaxScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers , regularizers

In [5]:
df = pd.read_csv("data/train.csv")
df.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [6]:
df.columns

Index(['id', 'age', 'daily_screen_time_hours', 'social_media_hours',
       'gaming_hours', 'work_study_hours', 'sleep_hours',
       'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
       'gender', 'stress_level', 'academic_work_impact', 'addicted_label'],
      dtype='object')

In [7]:
df["gender"].replace({"Male":0 , "Female":1 , "Other":2} , inplace=True)
df["stress_level"].replace({"Medium":0 , "Low":1 , "High":2} , inplace=True)
df["academic_work_impact"].replace({"No":0 , "Yes":1} , inplace=True)

C:\Users\Mesbah\AppData\Local\Temp\ipykernel_8596\3612125029.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["gender"].replace({"Male":0 , "Female":1 , "Other":2} , inplace=True)
C:\Users\Mesbah\AppData\Local\Temp\ipykernel_8596\3612125029.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d

In [8]:
df.loc[
    (df["age"] < 5) | (df["age"] > 100),
    "age"
] = None

df.loc[
    (df["sleep_hours"] < 0) | (df["sleep_hours"] > 24),
    "sleep_hours"
] = None

time_columns = [
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "weekend_screen_time"
]

for col in time_columns:
    df.loc[
        (df[col] < 0) | (df[col] > 24),
        col
    ] = None

In [9]:
for col in df.columns:
    df[col] = df[col].fillna(df[col].median())

In [10]:
df.drop(columns=["id"] , axis=0 , inplace=True)

In [11]:
x = df.iloc[: , 0:-1].values
y = df.iloc[: , -1].values

In [12]:
x

array([[24.  ,  7.77,  1.83, ...,  0.  ,  0.  ,  0.  ],
       [19.  ,  5.97,  1.08, ...,  1.  ,  0.  ,  0.  ],
       [18.  ,  5.09,  2.31, ...,  1.  ,  1.  ,  1.  ],
       ...,
       [27.  ,  7.77,  2.31, ...,  2.  ,  0.  ,  0.  ],
       [22.  , 11.78,  3.69, ...,  2.  ,  1.  ,  1.  ],
       [21.  ,  5.64,  1.03, ...,  2.  ,  0.  ,  1.  ]],
      shape=(691369, 12))

In [13]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.15, random_state=42
)

In [14]:
scaler = MinMaxScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


In [15]:
model = keras.Sequential(
    [
        layers.Input(shape=(x_train.shape[1],)),
        layers.Dense(
            128,
            kernel_initializer="he_normal",
            kernel_regularizer=regularizers.l2(0.001),
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Dropout(0.35),

        layers.Dense(
            128,
            kernel_initializer="he_normal",
            kernel_regularizer=regularizers.l2(0.001),
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        
        layers.Dropout(0.30),
        layers.Dense(
            64,
            kernel_initializer="he_normal",
            kernel_regularizer=regularizers.l2(0.001),
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Dropout(0.30),
        layers.Dense(
            64,
            kernel_initializer="he_normal",
            kernel_regularizer=regularizers.l2(0.001),
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Dropout(0.30),

        layers.Dense(
            32,
            kernel_initializer="he_normal",
            kernel_regularizer=regularizers.l2(0.001),
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Dropout(0.25),
        layers.Dense(
            32,
            kernel_initializer="he_normal",
            kernel_regularizer=regularizers.l2(0.001),
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.Dropout(0.25),

        layers.Dense(
            16,
            kernel_initializer="he_normal",
            kernel_regularizer=regularizers.l2(0.001),
        ),
        layers.Activation("relu"),
        layers.Dense(
            16,
            kernel_initializer="he_normal",
            kernel_regularizer=regularizers.l2(0.001),
        ),
        layers.Activation("relu"),

        layers.Dense(1, activation="sigmoid"),
    ]
)

In [16]:
model.compile(
     optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

In [17]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=20,
    restore_best_weights=True,
    verbose=1
)


reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=7,
    min_lr=1e-6,
    verbose=1
)

In [18]:
history = model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=64,
    verbose=1,
    callbacks=[
        early_stopping,
        reduce_lr
    ],
)

Epoch 1/50
7346/7346 ━━━━━━━━━━━━━━━━━━━━ 27s 3ms/step - accuracy: 0.8233 - auc: 0.8959 - loss: 0.5135 - precision: 0.8721 - recall: 0.8802 - val_accuracy: 0.8451 - val_auc: 0.9229 - val_loss: 0.3365 - val_precision: 0.8914 - val_recall: 0.8905 - learning_rate: 0.0010
Epoch 2/50
7346/7346 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - accuracy: 0.8349 - auc: 0.9103 - loss: 0.3576 - precision: 0.8783 - recall: 0.8908 - val_accuracy: 0.8460 - val_auc: 0.9243 - val_loss: 0.3381 - val_precision: 0.9031 - val_recall: 0.8774 - learning_rate: 0.0010
Epoch 3/50
7346/7346 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - accuracy: 0.8364 - auc: 0.9111 - loss: 0.3552 - precision: 0.8787 - recall: 0.8928 - val_accuracy: 0.8453 - val_auc: 0.9267 - val_loss: 0.3352 - val_precision: 0.9198 - val_recall: 0.8569 - learning_rate: 0.0010
Epoch 4/50
7346/7346 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - accuracy: 0.8377 - auc: 0.9119 - loss: 0.3533 - precision: 0.8778 - recall: 0.8961 - val_accuracy: 0.8494 - val_auc: 0.9268 - val_loss: 

KeyboardInterrupt: 

In [ ]:
results = model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print("Test Loss:", results[0])
print("Test Accuracy:", results[1])
print("Test Precision:", results[2])
print("Test Recall:", results[3])
print("Test AUC:", results[4])

NameError: name 'model' is not defined

In [ ]:

predictions = model.predict(x_test)

print(predictions[:10])